# Конвертация landing → bronze (parquet)

CSV из `data/landing` переводятся в parquet в `data/bronze`. Типы колонок объявлены явно в `scripts/parquet_convert.py` (`TABLES`), источник — `docs_favorita/0_raw_data_exploration/data_dictionary.md`. Строки сортируются по ключу таблицы, сжатие — zstd. Данные не очищаются: только приведение к типам.

`sample_submission.csv` не конвертируется — это шаблон файла сабмита, а не данные.

## Подготовка

In [1]:
import duckdb
import pandas as pd
from scripts import parquet_convert as pc

Соединение in-memory: для конвертации файл базы не нужен, и так нет конфликта блокировки с `favorita.duckdb`, если он открыт в другой тетрадке. Ограничение памяти и каталог для сброса на диск нужны для сортировки train (~125 млн строк).

In [2]:
con = duckdb.connect()
con.sql("SET memory_limit = '8GB'")  # оставить память Windows и Jupyter
con.sql("SET temp_directory = 'data/tmp'")  # куда сбрасывать при нехватке RAM
con.sql("SET preserve_insertion_order = false");  # ORDER BY при этом соблюдается

Объявленные схемы.

In [3]:
pd.DataFrame(
    [
        {"table": t, "column": c, "type": ty, "sort_key": c in spec.sort_by}
        for t, spec in pc.TABLES.items()
        for c, ty in spec.columns.items()
    ]
)

,table,column,type,sort_key
0,train,id,INTEGER,False
1,train,date,DATE,True
2,train,store_nbr,SMALLINT,True
3,train,item_nbr,INTEGER,True
4,train,unit_sales,DOUBLE,False
5,train,onpromotion,BOOLEAN,False
6,test,id,INTEGER,False
7,test,date,DATE,True
8,test,store_nbr,SMALLINT,True
9,test,item_nbr,INTEGER,True


## Конвертация

Сначала маленькие таблицы — быстро и сразу видно, если схема где-то не сходится с данными.

In [4]:
small_tables = [t for t in pc.TABLES if t != "train"]
pc.convert_all(con, small_tables)

{'test': WindowsPath('data/bronze/test.parquet'),
 'stores': WindowsPath('data/bronze/stores.parquet'),
 'items': WindowsPath('data/bronze/items.parquet'),
 'transactions': WindowsPath('data/bronze/transactions.parquet'),
 'oil': WindowsPath('data/bronze/oil.parquet'),
 'holidays_events': WindowsPath('data/bronze/holidays_events.parquet')}

train — самый долгий шаг: чтение ~5 ГБ CSV и внешняя сортировка со сбросом на диск в `data/tmp`.

In [5]:
pc.convert_table(con, "train")

WindowsPath('data/bronze/train.parquet')

## Проверка

Для каждой таблицы сравниваются схема, число строк, число NULL по колонкам (против сырого CSV), отпечаток содержимого и порядок строк. Подробности — в docstring `verify_table`. Для train проверка занимает несколько проходов по данным.

In [6]:
reports = pc.verify_all(con)
checks = pd.concat([r.to_frame() for r in reports], ignore_index=True)
checks

,table,check,expected,actual,passed
0,train,schema,"[('id', 'INTEGER'), ('date', 'DATE'), ('store_...","[('id', 'INTEGER'), ('date', 'DATE'), ('store_...",True
1,train,rows,125497040,125497040,True
2,train,nulls.id,0,0,True
3,train,nulls.date,0,0,True
4,train,nulls.store_nbr,0,0,True
5,train,nulls.item_nbr,0,0,True
6,train,nulls.unit_sales,0,0,True
7,train,nulls.onpromotion,21657651,21657651,True
8,train,content_hash,1157532809165435244229409293,1157532809165435244229409293,True
9,train,sorted,0,0,True


In [7]:
checks[~checks["passed"]]

,table,check,expected,actual,passed


In [8]:
for r in reports:
    r.raise_if_failed()
print("Все проверки пройдены")

Все проверки пройдены


## Размер файлов

In [9]:
pd.DataFrame(
    [
        {
            "table": t,
            "csv_mb": (pc.LANDING_DIR / f"{t}.csv").stat().st_size / 2**20,
            "parquet_mb": (pc.BRONZE_DIR / f"{t}.parquet").stat().st_size / 2**20,
        }
        for t in pc.TABLES
    ]
).assign(ratio=lambda df: df["csv_mb"] / df["parquet_mb"]).round(2)

,table,csv_mb,parquet_mb,ratio
0,train,4765.94,586.83,8.12
1,test,120.32,9.93,12.11
2,stores,0.00,0.00,0.92
3,items,0.10,0.02,4.65
4,transactions,1.48,0.16,9.21
5,oil,0.02,0.01,3.08
6,holidays_events,0.02,0.00,5.81


In [10]:
con.close()